**NOTE** `phimats` environment should be used as kernel

In [1]:
import numpy as np

from PreProcessing import PhysicsConfig, MeshConfig, PreProcessing as PP

from MeshManager import MeshManager

from BoundaryConditions import *

from PostProcessing import WriteXDMF

%load_ext autoreload
%autoreload 2

### Simulation data

In [24]:
SimulName = "SRB_Air"
# Element sets
nElementSets = 1
# Number of steps to achieve the load
nSteps = 400

### Read mesh file

In [ ]:
# Element name
elementName = "quad"  		# meshio compatible element name
mesh = MeshManager("../SmoothAxi.msh", elementName)
mesh.WriteMesh("../SmoothAxi")

In [26]:
# Create the config object
meshConfig = MeshConfig(
    nTotNodes=mesh.get_nTotNodes(),
    nTotElements=mesh.get_nTotElements(),
    nDim=mesh.get_nDim(),
    materialNames=mesh.getMaterialNames(),
)

### Apply mechanics boundary conditions
**NOTE** This is the total load to be achieved in `nSteps`.

In [ ]:
dl = 30e-3*0.28 # [m]

In [ ]:
# Dirichlet BCs list
presBCs = []

# Top pull
presBCs.extend(AssignDirichletBC(mesh.getNodesByGroup("Top"), dof=1, value=dl))

# Bottom y=0
presBCs.extend(AssignDirichletBC(mesh.getNodesByGroup("Bottom"), dof=1, value=0.0))

# Bottom left corner pin (Fixed in X)
presBCs.extend(AssignDirichletBC(mesh.getNodesByGroup("Fix"), dof=0, value=0.0))

# Symmetry
presBCs.extend(AssignDirichletBC(mesh.getNodesByGroup("Symmetry_Axis"), dof=0, value=0.0))

# Write vtu for visualization 
WriteBCVTK(SimulName, mesh, presBCs, dofNames=['ux', 'uy'])

### Apply diffusion boundary conditions

### Mechanical material data

In [29]:
# Analysis type
AnalysisType = "AxiSymmetricPFF"
# Isotropy
Isotropy = "Isotropic"
# Young's modulus
Emod = 210e9
# Poisson's ratio
nu = 0.3

# Plasticity type
Plasticity = "IsoHard"
HardeningLaw = "Voce"
sig_y0 = 533e6    # Yield strength (Pa)
sig_sat = 685e6   # Saturation stress (Pa)
H0 = 9000e6       # Initial hardening rate (Pa)

rho_0 = 1e11  # Initial dislocation density (m⁻²)
M = 3 		  # Taylor factor
alpha = 0.3   # Dislocation interaction constant
b = 2.5e-10   # Burgers vector (m)
G = Emod/(2*(1+nu)) # Shear modulus
k1 = 2*(G/(200*G*alpha*b))  # Multiplication coefficient
k2 = 10  # Recovery coefficient

MechMaterial = {
	"SmoothAxi" : {
		"Elastic" : {
			"AnalysisType" : AnalysisType,
			"Isotropy" 	 : Isotropy,
			"Emod" 		 : Emod,
			"nu"   		 : nu,
		},
       	"Plastic" : {
			"Plasticity"   : Plasticity,
			"HardeningLaw" : HardeningLaw,
			"sig_y0"       : sig_y0,
			"sig_sat"      : sig_sat,
			"H0"           : H0,
			"rho_0"        : rho_0,
			"M"            : M,
			"b"            : b,
			"alpha"        : alpha,
			"k1"           : k1,
   			"k2"           : k2,
		},
	},
}

### PFF material data

In [ ]:
const_ell = 3e-5     # Length-scale regularization parameter [m]
wc = 90e6 # Critical work density [J/m³]

Gc = 2*const_ell*wc	 # Fracture toughness [J/m²]

PFFMaterial = {
	"SmoothAxi" : {
		"PFF" : {
			"const_ell" : const_ell,
			"wc" 	    : wc,
		}
    }
}

print("l: ", const_ell, " m")
print("Gc: ", Gc, " J/m²")
print("wc: ", wc, " J/m³")

### Initialize simulation object

### Mechanical

In [31]:
# Create the config object
mechPhysConfig = PhysicsConfig(
    SimulName = SimulName,
    PhysicsType = "Mechanical",
    PhysicsCategory = "Plastic",
    nSteps    = nSteps,
    presBCs=presBCs
)

In [32]:
MechNotch = PP(mechPhysConfig, meshConfig, MechMaterial)


PHIMATS: SRB_Air | Mechanical (Plastic)
  Nodes/Elems : 30405 / 29996
  Total DOFs  : 60810
  BCs Count   : 320



In [33]:
MechNotch.WriteInputFile()

  Input file initialized: SRB_Air.mech.in.hdf5


#### PFF

In [34]:
# Create the config object
pffPhysConfig = PhysicsConfig(
    SimulName = SimulName,
    PhysicsType = "PFF",
    PhysicsCategory = list(PFFMaterial["SmoothAxi"].keys())[0],
    nSteps    = nSteps,
)

In [ ]:
PFFNotch = PP(pffPhysConfig, meshConfig, PFFMaterial)

In [36]:
PFFNotch.WriteInputFile()

  Input file initialized: SRB_Air.pff.in.hdf5


### Write output files

In [ ]:
OVERWRITE = False

In [ ]:
MechNotch.WriteOutputFile(overwrite=OVERWRITE)
PFFNotch.WriteOutputFile(overwrite=OVERWRITE)

The four components of `quadAxi` are:

1. **Radial Component ($r$):** Index `0`.
2. **Axial Component ($z$):** Index `1`.
3. **Hoop Component ($\theta$):** Acts in the circumferential direction. This is the "out-of-plane" normal stress/strain that arises due to the change in radius ($u_r/r$). This is index `2`.
4. **Shear Component ($rz$):** Represents the shear in the  plane. In your code, this is index `3`.

### Write xdmf

In [ ]:
WriteXDMF(SimulName,       # Base simulation name
          "../SmoothAxi",  # Mesh file 
          "quadAxi",       # Element name
          nSteps+1,        # Number of steps 
          components=["mech", "pff"],  # Physics
    	  nDim=2,  # Dimensions
    	  mechModel="Plastic", # Adds strain_p, stress_eq, etc.
    	)

**PETSc command line options**

```
./SRB_Air -snes_linesearch_type cp -snes_linesearch_damping 0.7 -snes_linesearch_monitor -snes_linesearch_max_it 50 -snes_max_it 100
```